# Fase 2 — Análise exploratória e seleção de variáveisEntrega da Fase 2 do planejamento: **Tabela 2** (descritivas + zero-inflação),**Figura 2** (correlação de Spearman) e o conjunto final de featuresdocumentado.Roda em cima de `data/processed/base_final.csv`, que é a saída do`python main.py`. Não reprocessa microdado nenhum.**Uma decisão de escopo, logo de cara:** todas as estatísticas aqui saem dosubconjunto que efetivamente vai para a clusterização — os **644** municípioscom nota IEGM, e não os 645 da base. A capital fica fora porque é fiscalizadapelo TCM-SP e não tem IEGM. Descrever 645 e clusterizar 644 produziria umaTabela 2 que não descreve o que foi modelado.

In [ ]:
import sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltRAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(RAIZ / "src"))from estilo import (aplicar_estilo, salvar, AZUL, LARANJA, MUDO, TINTA_2,                    CMAP_DIVERGENTE)from config import BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSEDaplicar_estilo()# Há mais de um Python instalado nesta máquina e nem todos têm as# dependências. Se der ImportError acima, o kernel do notebook está apontando# para o interpretador errado -- troque em "Select Kernel", no canto superior# direito, para o que este print mostrar quando funcionar.print(f"Python: {sys.executable}")base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})dic = pd.read_csv(DICIONARIO_CSV)print(f"base_final: {base.shape[0]} municípios x {base.shape[1]} colunas")

## 1. O espaço de features vem do dicionário, não de uma lista na mão`dicionario_base.csv` diz o bloco e o papel de cada coluna. Ler dali é o quepermite pesar os blocos na Fase 3 e o que garante que a Tabela 2 e aclusterização falem exatamente das mesmas variáveis. Se alguém acrescentar umafeature no `merge_bases.py`, ela aparece aqui sozinha.

In [ ]:
features = dic.loc[dic["papel"] == "feature", "coluna"].tolist()bloco_de = dic.set_index("coluna")["bloco"]# Ordem por bloco: é assim que a matriz de correlação fica legível.ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]features = sorted(features, key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))modelagem = base[~base["flag_sem_iegm"]].copy()print(f"{len(features)} features, {len(modelagem)} municípios na modelagem")print(pd.Series([bloco_de[c] for c in features]).value_counts()        .reindex(ORDEM_BLOCOS).to_string())print("\nExcluído por falta de IEGM:",      base.loc[base["flag_sem_iegm"], "municipio"].tolist())

## 2. Tabela 2 — descritivas das taxas e zero-inflaçãoA coluna `p_zeros` é o número que decide o desenho: uma taxa com 30% de zerosnão é uma variável contínua mal comportada, é uma mistura de "não aconteceu"com "aconteceu tanto". Ela reaparece na §4.

In [ ]:
taxas = [c for c in features         if c.startswith("taxa_") and c != "taxa_urbanizacao"]tabela2 = pd.DataFrame({    "média": modelagem[taxas].mean(),    "mediana": modelagem[taxas].median(),    "desvio": modelagem[taxas].std(),    "mín": modelagem[taxas].min(),    "máx": modelagem[taxas].max(),    "% zeros": (modelagem[taxas] == 0).mean() * 100,    "assimetria": modelagem[taxas].skew(),}).round(2)tabela2.to_csv(DATA_PROCESSED / "tabela2_descritivas.csv")tabela2

Duas leituras que valem parágrafo no artigo:- **A média é maior que a mediana em todas as taxas** — a assinatura da  assimetria à direita. É o que justifica `log1p` antes de qualquer distância  euclidiana.- **Os máximos não são as cidades mais violentas, são as menores.** Confiram  quem são: a taxa é por 100 mil habitantes e o denominador de um município de  2 mil pessoas transforma uma ocorrência em um valor enorme. É o problema dos  números pequenos, marcado na base como `flag_pop_pequena`.

In [ ]:
extremos = []for c in taxas:    linha = modelagem.loc[modelagem[c].idxmax()]    extremos.append({        "taxa": c,        "máximo": round(linha[c], 1),        "município": linha["municipio"],        "população": int(linha["populacao"]),        "pop < 5.000": bool(linha["flag_pop_pequena"]),    })pd.DataFrame(extremos)

## 3. Distribuições antes e depois de `log1p`Coluna da esquerda: a taxa como está. Coluna da direita: depois de `log1p`.A comparação é feita em **painéis lado a lado, não por cor** — assim a figuracontinua legível em escala de cinza, que é requisito da entrega.

In [ ]:
alvos = taxas + ["pib_percapita"]fig, axes = plt.subplots(len(alvos), 2, figsize=(7.2, 1.35 * len(alvos)))for i, col in enumerate(alvos):    s = modelagem[col].dropna()    axes[i, 0].hist(s, bins=40, color=AZUL, edgecolor=None)    axes[i, 1].hist(np.log1p(s), bins=40, color=LARANJA, edgecolor=None)    axes[i, 0].set_ylabel(col.replace("taxa_", ""), rotation=0,                          ha="right", va="center", fontsize=8)    for j, rotulo in enumerate(["bruta", "log1p"]):        ax = axes[i, j]        ax.set_yticks([])        ax.grid(False)        ax.tick_params(labelsize=7)        sk = (s if j == 0 else np.log1p(s)).skew()        ax.text(0.97, 0.85, f"assim. {sk:+.2f}", transform=ax.transAxes,                ha="right", va="top", fontsize=7, color=TINTA_2)        if i == 0:            ax.set_title(rotulo)fig.suptitle("Distribuições antes e depois de log1p (n=644)", y=1.005)fig.tight_layout()salvar(fig, "figura_distribuicoes_log1p")plt.show()

## 4. O que `log1p` resolve — e o que ele não resolveOlhando a coluna da direita acima, várias taxas ficaram com assimetria**negativa** depois da transformação. Isso parece excesso de correção, mas nãoé: é a zero-inflação reaparecendo do outro lado.`log1p` manda os zeros para 0 enquanto o corpo da distribuição vai para pertode `log(40) ≈ 3,7`. O resultado é uma cauda esquerda que não existia. O testeabaixo separa as duas coisas — recalcula a assimetria pós-`log1p` ignorando oszeros:

In [ ]:
diag = []for c in taxas:    s = modelagem[c]    diag.append({        "taxa": c,        "% zeros": round((s == 0).mean() * 100, 1),        "assim. bruta": round(s.skew(), 2),        "assim. log1p": round(np.log1p(s).skew(), 2),        "assim. log1p sem zeros": round(np.log1p(s[s > 0]).skew(), 2),    })pd.DataFrame(diag)

Retirando os zeros, toda assimetria pós-`log1p` cai para a faixaaproximada de −0,5 a +0,7 — praticamente simétrica. O caso mais eloquente é`taxa_estupro_total`: **7 municípios com zero** puxam a assimetria para pertode −2,3.**Conclusão para a Fase 3:** `log1p` está fazendo o trabalho dele sobre ocorpo da distribuição, e deve ser mantido. O que sobra é zero-inflação, que éoutro problema e pede outro remédio — encolhimento bayesiano das taxas, ou aanálise de sensibilidade excluindo os municípios de `flag_pop_pequena`. Nãoadianta trocar de transformação.

## 5. Figura 2 — correlação de SpearmanSpearman e não Pearson: as taxas não são normais e Pearson mede relaçãolinear. As variáveis estão ordenadas por bloco, com as divisórias marcadas —assim dá para ver se os blocos são redundantes entre si ou complementares, queé a premissa de integrar as três fontes.Só as células com |ρ| ≥ 0,5 vêm rotuladas. Isso mantém a figura limpa e, dequebra, preserva o sinal da correlação em escala de cinza, onde os dois polosda divergente ficam parecidos.

In [ ]:
rho = modelagem[features].corr(method="spearman")rotulos = [c.replace("taxa_", "").replace("_ord", "") for c in features]# k=0 esconde também a diagonal: ela é sempre 1, não informa nada e o# vermelho sólido puxaria o olho para o único lugar sem informação.mascara = np.triu(np.ones_like(rho, dtype=bool), k=0)plot = rho.mask(mascara)fig, ax = plt.subplots(figsize=(8.4, 7.4))im = ax.imshow(plot, cmap=CMAP_DIVERGENTE, vmin=-1, vmax=1)ax.set_xticks(range(len(features)), rotulos, rotation=90, fontsize=7.5)ax.set_yticks(range(len(features)), rotulos, fontsize=7.5)ax.grid(False)for lado in ax.spines.values():    lado.set_visible(False)# Divisórias entre blocos.corte = 0for b in ORDEM_BLOCOS[:-1]:    corte += sum(1 for c in features if bloco_de[c] == b)    ax.axhline(corte - 0.5, color=MUDO, lw=1.0)    ax.axvline(corte - 0.5, color=MUDO, lw=1.0)for i in range(len(features)):    for j in range(i + 1):        v = rho.iat[i, j]        if i != j and abs(v) >= 0.5:            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,                    color="white" if abs(v) > 0.75 else "#0b0b0b")cb = fig.colorbar(im, ax=ax, shrink=0.62, ticks=[-1, -0.5, 0, 0.5, 1])cb.set_label("ρ de Spearman", fontsize=8)cb.outline.set_visible(False)ax.set_title("Correlação de Spearman entre as 19 features (n=644)", pad=12)fig.tight_layout()salvar(fig, "figura2_spearman")plt.show()

## 6. Poda por redundânciaO critério do planejamento: onde |ρ| > 0,85, fundir ou descartar, registrandoo que saiu e por quê.

In [ ]:
LIMIAR = 0.85pares = [    {"a": a, "b": b, "rho": round(rho.loc[a, b], 3),     "bloco_a": bloco_de[a], "bloco_b": bloco_de[b]}    for i, a in enumerate(features) for b in features[i + 1:]    if abs(rho.loc[a, b]) > LIMIAR]if pares:    display(pd.DataFrame(pares).sort_values("rho", key=abs, ascending=False))else:    print(f"Nenhum par acima de |rho| > {LIMIAR}. Nada a podar:")    print("as 19 features seguem inteiras para a Fase 3.")maior = (rho.where(~np.eye(len(features), dtype=bool)).abs().stack().idxmax())print(f"\nMaior correlação em módulo: {maior[0]} x {maior[1]} = "      f"{rho.loc[maior[0], maior[1]]:+.3f}")

In [ ]:
mais_fortes = (rho.where(np.tril(np.ones_like(rho, dtype=bool), k=-1))                  .stack().rename("rho").reset_index()                  .rename(columns={"level_0": "a", "level_1": "b"}))mais_fortes["|rho|"] = mais_fortes["rho"].abs()mais_fortes.sort_values("|rho|", ascending=False).head(12).round(3)

## 7. O que fica decidido1. **Conjunto de features: as 19 originais, sem poda.** Nenhum par passa de   |ρ| > 0,85 — o máximo é `roubo_outros × roubo_veiculo`, em torno de 0,76,   longe do limiar. Isso é resultado a reportar, não ausência de resultado:   significa que as nove taxas medem fenômenos distintos e que os três blocos   não são redundantes entre si, que é a premissa de integrá-los numa etapa   só.2. **`log1p` nas taxas e no PIB per capita, mantido**, pelo que a §4 mostrou.3. **A zero-inflação é problema separado** e vai para a análise de   sensibilidade da Fase 4, não para a escolha de transformação.4. **Tabela 2 gravada** em `data/processed/tabela2_descritivas.csv`;   **Figura 2** em `figuras/figura2_spearman.png`.